<a href="https://colab.research.google.com/github/mmbc560/GUIA2/blob/main/Laboratorio_Sesion8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ================================================================
# MACHINE LEARNING APLICADO A EXCELENCIA OPERACIONAL
# LABORATORIO INTEGRAL CON INTERPRETACIONES GERENCIALES
#
# PCA + K-MEANS + BENCHMARKING + GENERACIÓN DE VALOR
#
# Diseñado para Google Colab
# ================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import unicodedata
import re
import warnings

from google.colab import files
from IPython.display import display, Markdown
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titlesize"] = 13

SEMILLA = 42
INDICADORES = ["ROA", "ROE", "EBIT", "MN"]

# Si se desea fijar K manualmente, reemplazar None
# por un número entero, por ejemplo 4.
K_MANUAL = None

# ================================================================
# FUNCIONES DE PRESENTACIÓN E INTERPRETACIÓN
# ================================================================

def titulo(texto):
    display(Markdown(f"## {texto}"))

def subtitulo(texto):
    display(Markdown(f"### {texto}"))

def interpretar(observacion, significado,
                excelencia, decision, precaucion=None):
    """
    Presenta una interpretación estructurada
    para apoyar la discusión en clase.
    """
    contenido = (
        f"**¿Qué muestran los resultados?** "
        f"{observacion}\n\n"
        f"**Interpretación gerencial:** "
        f"{significado}\n\n"
        f"**Perspectiva de excelencia operacional:** "
        f"{excelencia}\n\n"
        f"**Decisión o investigación recomendada:** "
        f"{decision}"
    )
    if precaucion:
        contenido += (
            f"\n\n**Precaución metodológica:** "
            f"{precaucion}"
        )
    display(Markdown(contenido))

def normalizar(texto):
    texto = unicodedata.normalize(
        "NFKD", str(texto)
    )
    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )
    return re.sub(
        r"[^A-Z0-9]", "", texto.upper()
    )

def detectar_columna(columnas, opciones):
    equivalencias = {
        normalizar(c): c for c in columnas
    }
    for opcion in opciones:
        clave = normalizar(opcion)
        if clave in equivalencias:
            return equivalencias[clave]
    return None

def elegir_columna(columnas, opciones,
                   etiqueta, obligatoria=False):
    encontrada = detectar_columna(
        columnas, opciones
    )
    if encontrada is not None:
        print(f"{etiqueta}: {encontrada}")
        return encontrada

    if not obligatoria:
        print(
            f"{etiqueta}: no identificada; "
            "se omitirá este análisis."
        )
        return None

    print(f"\nSelecciona la columna de {etiqueta}:")
    for i, col in enumerate(columnas):
        print(f"{i}: {col}")

    while True:
        respuesta = input(
            f"Número de columna para {etiqueta}: "
        )
        try:
            pos = int(respuesta)
            if 0 <= pos < len(columnas):
                return columnas[pos]
        except ValueError:
            pass
        print("Introduce un número válido.")

def convertir_numero(serie):
    """
    Convierte valores numéricos y textos con
    separadores decimales o porcentajes.
    """
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(
            serie, errors="coerce"
        )

    def limpiar(valor):
        if pd.isna(valor):
            return np.nan

        v = str(valor).strip()
        v = v.replace(" ", "").replace("%", "")

        if "," in v and "." in v:
            if v.rfind(",") > v.rfind("."):
                v = v.replace(".", "").replace(",", ".")
            else:
                v = v.replace(",", "")
        elif "," in v:
            v = v.replace(",", ".")

        try:
            return float(v)
        except ValueError:
            return np.nan

    return serie.map(limpiar).astype(float)

def texto_columna(serie):
    return (
        serie.astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

def describir_asimetria(valor):
    if valor > 1:
        return "asimetría positiva marcada"
    if valor > 0.5:
        return "asimetría positiva moderada"
    if valor < -1:
        return "asimetría negativa marcada"
    if valor < -0.5:
        return "asimetría negativa moderada"
    return "distribución relativamente simétrica"

# ================================================================
# 1. CARGA DEL EXCEL
# ================================================================

titulo("1. Carga de datos financieros")

print("Selecciona el archivo Excel:")
uploaded = files.upload()

nombre_archivo = list(uploaded.keys())[0]
libro = pd.ExcelFile(nombre_archivo)

print("\nHojas disponibles:", libro.sheet_names)

if len(libro.sheet_names) > 1:
    hoja = input(
        "Nombre de la hoja a analizar "
        "(Enter = primera): "
    ).strip()
    if not hoja:
        hoja = libro.sheet_names[0]
else:
    hoja = libro.sheet_names[0]

df = pd.read_excel(
    nombre_archivo, sheet_name=hoja
)

print("\nArchivo cargado correctamente")
print("Registros:", len(df))
print("Columnas:", len(df.columns))

display(df.head())

print("\nColumnas disponibles:")
for i, c in enumerate(df.columns):
    print(i, c)

# ================================================================
# 2. IDENTIFICACIÓN DE INDICADORES
# ================================================================

titulo("2. Identificación de indicadores financieros")

equivalencias = {
    "ROA": [
        "ROA", "RENTABILIDAD DEL ACTIVO",
        "RENTABILIDAD ACTIVOS", "RETURN ON ASSETS"
    ],
    "ROE": [
        "ROE", "RENTABILIDAD DEL PATRIMONIO",
        "RENTABILIDAD PATRIMONIO", "RETURN ON EQUITY"
    ],
    "EBIT": [
        "EBIT", "MARGEN EBIT",
        "RESULTADO OPERACIONAL",
        "UTILIDAD OPERACIONAL"
    ],
    "MN": [
        "MN", "MARGEN NETO",
        "MARGEN DE UTILIDAD NETA",
        "NET MARGIN"
    ]
}

mapa = {}

for indicador in INDICADORES:
    mapa[indicador] = elegir_columna(
        list(df.columns),
        equivalencias[indicador],
        indicador,
        obligatoria=True
    )

col_empresa = elegir_columna(
    list(df.columns),
    ["RAZON SOCIAL", "RAZÓN SOCIAL",
     "EMPRESA", "NOMBRE EMPRESA"],
    "Razón social"
)

col_ciiu = elegir_columna(
    list(df.columns),
    ["CIIU", "CODIGO CIIU",
     "CÓDIGO CIIU", "ACTIVIDAD ECONOMICA"],
    "CIIU"
)

col_ciudad = elegir_columna(
    list(df.columns),
    ["CIUDAD", "CIUDAD DOMICILIO",
     "MUNICIPIO", "MUNICIPIO DOMICILIO"],
    "Ciudad"
)

col_departamento = elegir_columna(
    list(df.columns),
    ["DEPARTAMENTO",
     "DEPARTAMENTO DOMICILIO"],
    "Departamento"
)

# ================================================================
# 3. PREPARACIÓN DE DATOS
# ================================================================

titulo("3. Preparación y calidad de datos")

datos = pd.DataFrame(index=df.index)

for indicador in INDICADORES:
    datos[indicador] = convertir_numero(
        df[mapa[indicador]]
    )

datos = datos.replace(
    [np.inf, -np.inf], np.nan
)

faltantes = datos.isna().sum()

display(
    faltantes.to_frame("Datos faltantes")
)

total_original = len(datos)
datos = datos.dropna().copy()

excluidos = total_original - len(datos)
porcentaje_excluido = (
    excluidos / total_original * 100
    if total_original else 0
)

if len(datos) < 5:
    raise ValueError(
        "Se necesitan al menos cinco registros "
        "válidos para este laboratorio."
    )

if any(
    datos[c].std() == 0
    for c in INDICADORES
):
    raise ValueError(
        "Existe un indicador sin variación. "
        "Revisa la selección de columnas."
    )

interpretar(
    observacion=(
        f"El archivo contiene {total_original:,} registros. "
        f"Se conservaron {len(datos):,} empresas "
        f"con información completa y se excluyeron "
        f"{excluidos:,} ({porcentaje_excluido:.2f}%)."
    ),
    significado=(
        "La calidad de los datos determina la "
        "confiabilidad de las comparaciones. "
        "Los valores faltantes pueden distorsionar "
        "los perfiles si no se tratan adecuadamente."
    ),
    excelencia=(
        "La calidad de la información constituye "
        "un requisito para la excelencia operacional. "
        "Una organización que no controla la "
        "integridad de sus indicadores puede "
        "tomar decisiones equivocadas sobre "
        "productividad, rentabilidad y capacidad."
    ),
    decision=(
        "Revisar por qué existen registros incompletos "
        "y establecer controles de captura, "
        "validación y trazabilidad de datos."
    ),
    precaucion=(
        "La exclusión de registros puede introducir "
        "sesgo si las empresas con datos faltantes "
        "son sistemáticamente diferentes."
    )
)

# ================================================================
# 4. ANÁLISIS DESCRIPTIVO
# ================================================================

titulo("4. Estadísticas descriptivas")

estadisticas = datos.describe().T
display(estadisticas.round(4))

fig, axes = plt.subplots(
    2, 2, figsize=(14, 10)
)

for ax, indicador in zip(
    axes.ravel(), INDICADORES
):
    sns.histplot(
        datos[indicador],
        kde=True,
        color="#167D9D",
        ax=ax
    )
    ax.axvline(
        datos[indicador].mean(),
        color="red",
        linestyle="--",
        label="Media"
    )
    ax.axvline(
        datos[indicador].median(),
        color="green",
        linestyle=":",
        label="Mediana"
    )
    ax.set_title(
        f"Distribución de {indicador}"
    )
    ax.legend()

plt.suptitle(
    "Distribución de indicadores financieros",
    fontsize=16,
    fontweight="bold"
)
plt.tight_layout()
plt.show()

for indicador in INDICADORES:

    media = datos[indicador].mean()
    mediana = datos[indicador].median()
    desv = datos[indicador].std()
    asimetria = datos[indicador].skew()
    p10 = datos[indicador].quantile(0.10)
    p90 = datos[indicador].quantile(0.90)

    interpretar(
        observacion=(
            f"**{indicador}:** media {media:.4f}, "
            f"mediana {mediana:.4f}, "
            f"desviación estándar {desv:.4f}. "
            f"El percentil 10 es {p10:.4f} y "
            f"el percentil 90 es {p90:.4f}. "
            f"La distribución presenta "
            f"{describir_asimetria(asimetria)} "
            f"(asimetría = {asimetria:.2f})."
        ),
        significado=(
            "La comparación entre media, mediana "
            "y percentiles permite identificar "
            "heterogeneidad financiera y posibles "
            "empresas con resultados extremos. "
            "Una media elevada no implica que "
            "la mayoría de las empresas tenga "
            "ese desempeño."
        ),
        excelencia=(
            "La dispersión financiera plantea "
            "preguntas sobre diferencias en "
            "productividad, utilización de activos, "
            "estructura de costos y gestión "
            "de recursos. Sin embargo, esas "
            "causas todavía no están demostradas."
        ),
        decision=(
            "Seleccionar empresas de los percentiles "
            "superiores e inferiores para investigar "
            "sus procesos, estructura de costos, "
            "capacidad instalada y prácticas de gestión."
        )
    )

# ================================================================
# 5. BOXPLOTS Y VALORES EXTREMOS
# ================================================================

titulo("5. Dispersión y valores extremos")

fig, axes = plt.subplots(
    2, 2, figsize=(14, 9)
)

for ax, indicador in zip(
    axes.ravel(), INDICADORES
):
    sns.boxplot(
        y=datos[indicador],
        color="#79B7A3",
        ax=ax
    )
    ax.set_title(
        f"Boxplot de {indicador}"
    )

plt.tight_layout()
plt.show()

for indicador in INDICADORES:

    q1 = datos[indicador].quantile(0.25)
    q3 = datos[indicador].quantile(0.75)
    iqr = q3 - q1

    inferior = q1 - 1.5*iqr
    superior = q3 + 1.5*iqr

    atipicos = (
        (datos[indicador] < inferior)
        | (datos[indicador] > superior)
    )

    cantidad = int(atipicos.sum())

    interpretar(
        observacion=(
            f"En {indicador} se identificaron "
            f"{cantidad:,} observaciones fuera "
            f"de los límites convencionales "
            f"del boxplot, equivalentes al "
            f"{cantidad/len(datos)*100:.2f}% "
            "de los registros."
        ),
        significado=(
            "Estos valores pueden corresponder "
            "a empresas con perfiles financieros "
            "inusuales, diferencias de escala, "
            "eventos extraordinarios o errores "
            "de registro."
        ),
        excelencia=(
            "Los casos extremos pueden servir "
            "para investigar prácticas destacadas "
            "o situaciones críticas. En Lean "
            "y Six Sigma, distinguir variación "
            "habitual de comportamientos especiales "
            "es importante para priorizar "
            "las investigaciones."
        ),
        decision=(
            "Examinar la calidad de los registros "
            "y comparar las empresas extremas "
            "con organizaciones de actividad "
            "económica y tamaño semejantes."
        ),
        precaucion=(
            "Un valor atípico financiero no es "
            "automáticamente un defecto operacional "
            "ni debe eliminarse sin justificación."
        )
    )

# ================================================================
# 6. CORRELACIONES
# ================================================================

titulo("6. Correlación entre indicadores")

corr = datos.corr()

display(corr.round(3))

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr,
    annot=True,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    fmt=".2f",
    linewidths=1
)
plt.title(
    "Matriz de correlación financiera",
    fontweight="bold"
)
plt.tight_layout()
plt.show()

pares = [
    (a, b, corr.loc[a, b])
    for i, a in enumerate(INDICADORES)
    for b in INDICADORES[i+1:]
]

par_fuerte = max(
    pares, key=lambda x: abs(x[2])
)

a, b, r = par_fuerte

interpretar(
    observacion=(
        f"La asociación lineal más intensa "
        f"en valor absoluto se presenta entre "
        f"{a} y {b}, con r = {r:.3f}. "
        f"Su dirección es "
        f"{'positiva' if r >= 0 else 'negativa'}."
    ),
    significado=(
        "Cuando dos indicadores presentan "
        "una relación intensa, parte de la "
        "información que aportan puede "
        "ser redundante. Esto es relevante "
        "porque el análisis PCA busca "
        "resumir patrones compartidos."
    ),
    excelencia=(
        "La relación entre rentabilidad, "
        "resultados operacionales y márgenes "
        "puede orientar hipótesis sobre "
        "eficiencia en el uso de activos "
        "y capacidad para transformar "
        "ingresos en resultados financieros."
    ),
    decision=(
        "Investigar qué variables operacionales "
        "podrían explicar la relación observada: "
        "productividad, utilización, desperdicios, "
        "reprocesos y costos de no calidad."
    ),
    precaucion=(
        "La correlación no prueba causalidad. "
        "También puede estar influida por "
        "sector, tamaño, apalancamiento o "
        "condiciones del periodo."
    )
)

# ================================================================
# 7. ESTANDARIZACIÓN
# ================================================================

titulo("7. Estandarización de indicadores")

scaler = StandardScaler()
X = scaler.fit_transform(datos)

datos_z = pd.DataFrame(
    X,
    columns=INDICADORES,
    index=datos.index
)

display(datos_z.head())

interpretar(
    observacion=(
        "Los cuatro indicadores se transformaron "
        "a puntuaciones estandarizadas, con "
        "media aproximada de cero y "
        "desviación estándar de uno."
    ),
    significado=(
        "Esto permite comparar variables "
        "que originalmente pueden tener "
        "magnitudes distintas."
    ),
    excelencia=(
        "En los sistemas de medición operacional "
        "es fundamental definir indicadores "
        "comparables. De lo contrario, "
        "una variable puede dominar el análisis "
        "por su escala y no por su importancia."
    ),
    decision=(
        "Verificar unidades, definiciones "
        "y periodos de los indicadores "
        "antes de aplicar modelos analíticos."
    )
)

# ================================================================
# 8. PCA COMPLETO
# ================================================================

titulo("8. PCA: reducción de dimensionalidad")

pca_completo = PCA()
pca_completo.fit(X)

varianza = (
    pca_completo.explained_variance_ratio_*100
)

tabla_varianza = pd.DataFrame({
    "Componente": [
        f"PC{i+1}" for i in range(4)
    ],
    "Varianza (%)": varianza,
    "Acumulada (%)": np.cumsum(varianza)
})

display(tabla_varianza.round(2))

fig, axes = plt.subplots(
    1, 2, figsize=(13, 5)
)

axes[0].bar(
    tabla_varianza["Componente"],
    tabla_varianza["Varianza (%)"],
    color=sns.color_palette("Set2", 4)
)
axes[0].set_ylabel("Varianza explicada (%)")
axes[0].set_title(
    "Información explicada por componente"
)

axes[1].plot(
    range(1, 5),
    tabla_varianza["Acumulada (%)"],
    marker="o",
    color="#167D9D"
)
axes[1].axhline(
    80,
    linestyle="--",
    color="red",
    label="Referencia: 80%"
)
axes[1].set_ylim(0, 105)
axes[1].set_xticks(range(1, 5))
axes[1].set_title("Varianza acumulada")
axes[1].legend()

plt.tight_layout()
plt.show()

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

varianza_2 = (
    pca.explained_variance_ratio_.sum()*100
)

resultado = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"],
    index=datos.index
)

interpretar(
    observacion=(
        f"PC1 explica {varianza[0]:.2f}% "
        f"de la variabilidad y PC2 explica "
        f"{varianza[1]:.2f}%. "
        f"Juntas conservan {varianza_2:.2f}% "
        "de la variabilidad de los cuatro "
        "indicadores estandarizados."
    ),
    significado=(
        "PCA transforma los indicadores "
        "originales en nuevas dimensiones "
        "que resumen sus patrones conjuntos. "
        "Permite estudiar una estructura "
        "financiera multidimensional "
        "mediante menos variables."
    ),
    excelencia=(
        "La reducción de dimensionalidad "
        "ayuda a simplificar tableros "
        "de control complejos y detectar "
        "patrones que podrían pasar "
        "desapercibidos al analizar "
        "cada indicador aisladamente."
    ),
    decision=(
        "Utilizar las componentes para "
        "visualizar perfiles empresariales "
        "y formular hipótesis sobre "
        "desempeño, sin sustituir "
        "los indicadores originales "
        "para las decisiones operacionales."
    ),
    precaucion=(
        "La varianza explicada mide información "
        "estadística conservada, no porcentaje "
        "de eficiencia operacional. "
        "Una representación con baja varianza "
        "acumulada puede ocultar diferencias."
    )
)

# ================================================================
# 9. CARGAS DEL PCA
# ================================================================

titulo("9. ¿Qué representan PC1 y PC2?")

cargas = pd.DataFrame(
    pca.components_.T,
    columns=["PC1", "PC2"],
    index=INDICADORES
)

display(cargas.round(4))

plt.figure(figsize=(8, 6))
sns.heatmap(
    cargas,
    annot=True,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    fmt=".3f"
)
plt.title(
    "Contribución de indicadores a las componentes"
)
plt.tight_layout()
plt.show()

for componente in ["PC1", "PC2"]:

    orden = (
        cargas[componente]
        .abs()
        .sort_values(ascending=False)
    )

    primero = orden.index[0]
    segundo = orden.index[1]

    interpretar(
        observacion=(
            f"En {componente}, los mayores pesos "
            f"absolutos corresponden a "
            f"{primero} ({cargas.loc[primero, componente]:.3f}) "
            f"y {segundo} "
            f"({cargas.loc[segundo, componente]:.3f})."
        ),
        significado=(
            "Estos indicadores tienen una "
            "participación importante en "
            "la construcción de la componente. "
            "Los signos permiten analizar "
            "si sus contribuciones se orientan "
            "en la misma dirección o en "
            "direcciones opuestas."
        ),
        excelencia=(
            "La componente puede ayudar "
            "a reconocer combinaciones de "
            "resultados financieros que merecen "
            "un diagnóstico de productividad, "
            "costos y utilización de recursos."
        ),
        decision=(
            "Analizar conjuntamente las cargas "
            "de todos los indicadores y contrastar "
            "sus patrones con información "
            "operacional de las empresas."
        ),
        precaucion=(
            "No debe asignarse automáticamente "
            "el nombre 'eficiencia operacional' "
            "a una componente financiera. "
            "El signo de una componente PCA "
            "también puede invertirse sin "
            "cambiar su significado estadístico."
        )
    )

# ================================================================
# 10. BIPLOT PCA
# ================================================================

titulo("10. Biplot de empresas e indicadores")

fig, ax = plt.subplots(figsize=(12, 8))

ax.scatter(
    resultado["PC1"],
    resultado["PC2"],
    alpha=0.25,
    s=22,
    color="#167D9D"
)

escala = max(
    np.percentile(np.abs(X_pca[:, 0]), 95),
    np.percentile(np.abs(X_pca[:, 1]), 95)
)

for i, indicador in enumerate(INDICADORES):

    dx = cargas.iloc[i, 0]*escala
    dy = cargas.iloc[i, 1]*escala

    ax.arrow(
        0, 0, dx, dy,
        color="#D95D39",
        width=0.015,
        head_width=0.12,
        length_includes_head=True
    )

    ax.text(
        dx*1.1, dy*1.1,
        indicador,
        fontsize=11,
        fontweight="bold"
    )

ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title(
    "Biplot PCA: perfiles financieros empresariales"
)
plt.tight_layout()
plt.show()

interpretar(
    observacion=(
        "Cada punto representa una empresa "
        "y su posición en las dos componentes. "
        "Las flechas representan las direcciones "
        "de las cargas de los indicadores."
    ),
    significado=(
        "Empresas cercanas en el plano "
        "presentan perfiles similares "
        "en las dimensiones representadas. "
        "Las empresas alejadas pueden "
        "tener estructuras financieras "
        "diferenciadas."
    ),
    excelencia=(
        "El gráfico facilita identificar "
        "organizaciones que podrían compararse "
        "mediante benchmarking. "
        "También permite detectar perfiles "
        "atípicos que requieren una "
        "investigación gerencial específica."
    ),
    decision=(
        "Seleccionar empresas próximas "
        "para comparar prácticas y empresas "
        "distantes para estudiar diferencias "
        "en costos, productividad, "
        "gestión de activos y riesgos."
    ),
    precaucion=(
        "La proximidad en PCA no demuestra "
        "similitud de procesos. "
        "Las distancias pueden estar "
        "distorsionadas si las dos componentes "
        "conservan poca variabilidad."
    )
)

# ================================================================
# 11. SELECCIÓN DE K
# ================================================================

titulo("11. Selección del número de clústeres")

max_k = min(8, len(X)-1)
valores_k = list(range(2, max_k+1))

inercias = []
silhouettes = []

for k in valores_k:

    modelo = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    etiquetas_temp = modelo.fit_predict(X)
    inercias.append(modelo.inertia_)

    if len(set(etiquetas_temp)) < 2:
        silhouettes.append(np.nan)
    else:
        silhouettes.append(
            silhouette_score(
                X,
                etiquetas_temp,
                sample_size=min(2500, len(X)),
                random_state=42
            )
        )

tabla_k = pd.DataFrame({
    "K": valores_k,
    "Inercia": inercias,
    "Silhouette": silhouettes
})

display(tabla_k.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(
    valores_k, inercias,
    marker="o", color="#167D9D"
)
axes[0].set_title("Método del codo")
axes[0].set_xlabel("Número de clústeres")
axes[0].set_ylabel("Inercia")

axes[1].plot(
    valores_k, silhouettes,
    marker="o", color="#D95D39"
)
axes[1].set_title("Coeficiente Silhouette")
axes[1].set_xlabel("Número de clústeres")
axes[1].set_ylabel("Silhouette")

plt.tight_layout()
plt.show()

if np.all(np.isnan(silhouettes)):
    raise ValueError(
        "No se obtuvo una segmentación válida."
    )

if K_MANUAL is not None:
    mejor_k = int(K_MANUAL)
    if not 2 <= mejor_k < len(X):
        raise ValueError("K_MANUAL no es válido.")
else:
    mejor_k = valores_k[
        int(np.nanargmax(silhouettes))
    ]

silhouette_elegido = silhouette_score(
    X,
    KMeans(
        n_clusters=mejor_k,
        random_state=42,
        n_init=10
    ).fit_predict(X),
    sample_size=min(2500, len(X)),
    random_state=42
)

interpretar(
    observacion=(
        f"El número seleccionado fue K={mejor_k}. "
        f"Su coeficiente Silhouette es "
        f"{silhouette_elegido:.3f}."
    ),
    significado=(
        "Silhouette compara la cohesión "
        "de las empresas dentro de su grupo "
        "con la separación respecto a otros "
        "grupos. Valores más altos indican "
        "una separación relativamente mejor "
        "para la segmentación evaluada."
    ),
    excelencia=(
        "Una segmentación clara permite "
        "diseñar estrategias diferenciadas "
        "de benchmarking, diagnóstico "
        "y priorización de mejoras."
    ),
    decision=(
        "Comprobar si los grupos obtenidos "
        "son suficientemente distintos "
        "y útiles para definir estrategias "
        "gerenciales específicas."
    ),
    precaucion=(
        "Un Silhouette alto no significa "
        "que las empresas sean más rentables "
        "o más eficientes. "
        "Evalúa la estructura de agrupamiento."
    )
)

# ================================================================
# 12. K-MEANS
# ================================================================

titulo("12. Segmentación empresarial con K-Means")

kmeans = KMeans(
    n_clusters=mejor_k,
    random_state=42,
    n_init=10
)

etiquetas = kmeans.fit_predict(X)+1

resultado["Cluster"] = etiquetas

perfil = datos.copy()
perfil["Cluster"] = etiquetas

conteos = (
    perfil["Cluster"]
    .value_counts()
    .sort_index()
)

display(
    conteos.to_frame("Número de empresas")
)

centros_pca = pca.transform(
    kmeans.cluster_centers_
)

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=resultado,
    x="PC1",
    y="PC2",
    hue="Cluster",
    palette="Set2",
    s=35,
    alpha=0.65
)

plt.scatter(
    centros_pca[:, 0],
    centros_pca[:, 1],
    marker="X",
    s=250,
    c="black",
    label="Centroides"
)

plt.title(
    "PCA + K-Means: segmentos financieros",
    fontweight="bold"
)
plt.tight_layout()
plt.show()

mayor_cluster = conteos.idxmax()
menor_cluster = conteos.idxmin()

interpretar(
    observacion=(
        f"Se identificaron {mejor_k} grupos. "
        f"El clúster {mayor_cluster} es el "
        f"más numeroso, con "
        f"{conteos.max():,} empresas "
        f"({conteos.max()/len(perfil)*100:.1f}%). "
        f"El clúster {menor_cluster} contiene "
        f"{conteos.min():,} empresas."
    ),
    significado=(
        "La segmentación revela que las "
        "empresas no presentan un único "
        "perfil financiero. "
        "Existen grupos con combinaciones "
        "diferentes de rentabilidad "
        "y resultados."
    ),
    excelencia=(
        "Las estrategias de mejora no "
        "necesariamente deben ser iguales "
        "para todas las empresas. "
        "Los perfiles pueden ayudar "
        "a diseñar diagnósticos y programas "
        "de mejora diferenciados."
    ),
    decision=(
        "Caracterizar cada clúster, "
        "identificar brechas financieras "
        "y formular hipótesis sobre "
        "los procesos que podrían "
        "explicar esas diferencias."
    ),
    precaucion=(
        "Los clústeres se formaron con "
        "indicadores financieros, no con "
        "mediciones directas de procesos. "
        "Sus números son etiquetas arbitrarias."
    )
)

# ================================================================
# 13. PERFILES FINANCIEROS
# ================================================================

titulo("13. Comparación financiera de clústeres")

medias = perfil.groupby("Cluster")[INDICADORES].mean()
medianas = perfil.groupby("Cluster")[INDICADORES].median()

display(medias.round(4))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, indicador in zip(
    axes.ravel(), INDICADORES
):
    sns.barplot(
        data=perfil,
        x="Cluster",
        y=indicador,
        hue="Cluster",
        palette="Set2",
        errorbar=None,
        legend=False,
        ax=ax
    )
    ax.set_title(
        f"Promedio de {indicador} por clúster"
    )

plt.suptitle(
    "Brechas financieras entre clústeres",
    fontsize=16,
    fontweight="bold"
)
plt.tight_layout()
plt.show()

for indicador in INDICADORES:

    mejor = medias[indicador].idxmax()
    menor = medias[indicador].idxmin()

    valor_mejor = medias.loc[mejor, indicador]
    valor_menor = medias.loc[menor, indicador]

    interpretar(
        observacion=(
            f"En {indicador}, el mayor promedio "
            f"corresponde al clúster {mejor} "
            f"({valor_mejor:.4f}) y el menor "
            f"al clúster {menor} "
            f"({valor_menor:.4f}). "
            f"La brecha absoluta es "
            f"{valor_mejor-valor_menor:.4f}."
        ),
        significado=(
            "La diferencia refleja perfiles "
            "financieros distintos entre "
            "los segmentos identificados."
        ),
        excelencia=(
            "La brecha puede orientar "
            "investigaciones sobre "
            "productividad, utilización "
            "de activos, costos operacionales, "
            "reprocesos y gestión de recursos."
        ),
        decision=(
            "Comparar empresas del mismo "
            "sector y tamaño, investigar "
            "prácticas de gestión y "
            "establecer indicadores "
            "operacionales que permitan "
            "explicar la brecha."
        ),
        precaucion=(
            "Las diferencias entre clústeres "
            "no son una validación independiente: "
            "los grupos fueron construidos "
            "utilizando estos indicadores."
        )
    )

# ================================================================
# 14. HEATMAP DE PERFILES
# ================================================================

titulo("14. Heatmap de fortalezas y debilidades")

perfil_z = datos_z.copy()
perfil_z["Cluster"] = etiquetas

medias_z = (
    perfil_z.groupby("Cluster")[INDICADORES].mean()
)

plt.figure(figsize=(10, 6))
sns.heatmap(
    medias_z,
    annot=True,
    cmap="RdYlGn",
    center=0,
    fmt=".2f",
    linewidths=1
)
plt.title(
    "Perfil relativo de indicadores por clúster"
)
plt.tight_layout()
plt.show()

for cluster in medias_z.index:

    fila = medias_z.loc[cluster]

    fortaleza = fila.idxmax()
    debilidad = fila.idxmin()

    interpretar(
        observacion=(
            f"En el clúster {cluster}, "
            f"el indicador relativamente "
            f"más alto es {fortaleza} "
            f"({fila[fortaleza]:.2f} desviaciones "
            f"estándar respecto a la media), "
            f"mientras que el más bajo "
            f"es {debilidad} "
            f"({fila[debilidad]:.2f})."
        ),
        significado=(
            "El mapa permite reconocer "
            "qué indicadores diferencian "
            "a cada segmento frente "
            "al promedio de la muestra."
        ),
        excelencia=(
            "Una empresa puede mostrar "
            "resultados favorables en "
            "algunas dimensiones y "
            "desfavorables en otras. "
            "La excelencia operacional "
            "requiere una visión equilibrada "
            "y no solamente optimizar "
            "un indicador financiero."
        ),
        decision=(
            f"Investigar las causas "
            f"del desempeño relativo en "
            f"{debilidad} y contrastarlas "
            "con prácticas de empresas "
            "comparables."
        ),
        precaucion=(
            "El indicador relativamente "
            "más bajo no necesariamente "
            "es negativo en términos absolutos, "
            "ni el más alto demuestra "
            "una fortaleza operacional."
        )
    )

# ================================================================
# 15. ÍNDICE FINANCIERO
# ================================================================

titulo("15. Posición financiera relativa")

# Índice exploratorio de percentiles.
# Los cuatro indicadores reciben igual peso.
# Se supone que valores mayores son preferibles.

percentiles = datos.rank(pct=True)*100
indice = percentiles.mean(axis=1)

perfil["Indice_Financiero"] = indice

ranking_cluster = (
    perfil.groupby("Cluster")["Indice_Financiero"]
    .mean()
    .sort_values(ascending=False)
)

display(
    ranking_cluster.to_frame(
        "Índice financiero promedio"
    ).round(2)
)

plt.figure(figsize=(10, 5))
sns.barplot(
    x=ranking_cluster.index.astype(str),
    y=ranking_cluster.values,
    color="#167D9D"
)
plt.title(
    "Índice financiero relativo por clúster"
)
plt.xlabel("Clúster")
plt.ylabel("Índice promedio (0-100)")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

lider = ranking_cluster.index[0]
rezagado = ranking_cluster.index[-1]

interpretar(
    observacion=(
        f"El clúster {lider} registra "
        f"el índice financiero relativo "
        f"promedio más alto "
        f"({ranking_cluster.iloc[0]:.2f}). "
        f"El clúster {rezagado} presenta "
        f"el menor "
        f"({ranking_cluster.iloc[-1]:.2f})."
    ),
    significado=(
        "El índice resume la posición "
        "relativa de cada grupo "
        "considerando conjuntamente "
        "los cuatro indicadores."
    ),
    excelencia=(
        "Los grupos con mejores posiciones "
        "pueden proporcionar candidatos "
        "para benchmarking. "
        "Los grupos con menores posiciones "
        "pueden requerir diagnósticos "
        "de productividad, costos "
        "y utilización de activos."
    ),
    decision=(
        "Seleccionar empresas comparables "
        "y contrastar sus procesos, "
        "indicadores operacionales "
        "y prácticas de mejora continua."
    ),
    precaucion=(
        "Este índice no mide directamente "
        "excelencia operacional, solvencia "
        "ni capacidad de financiación. "
        "La interpretación depende de "
        "las definiciones y unidades "
        "de ROA, ROE, EBIT y MN."
    )
)

# ================================================================
# 16. INTEGRACIÓN EMPRESARIAL
# ================================================================

titulo("16. Integración con datos empresariales")

base = df.loc[datos.index].copy()

for indicador in INDICADORES:
    base[indicador] = datos[indicador]

base["Cluster"] = etiquetas
base["PC1"] = resultado["PC1"]
base["PC2"] = resultado["PC2"]
base["Indice_Financiero"] = indice

if col_empresa:
    base["Empresa"] = texto_columna(
        base[col_empresa]
    )
else:
    base["Empresa"] = [
        f"EMPRESA_{i+1}"
        for i in range(len(base))
    ]

if col_ciiu:
    base["CIIU_Analisis"] = texto_columna(
        base[col_ciiu]
    )
else:
    base["CIIU_Analisis"] = pd.NA

if col_ciudad:
    base["Ciudad_Analisis"] = (
        texto_columna(base[col_ciudad])
        .str.upper()
    )
else:
    base["Ciudad_Analisis"] = pd.NA

if col_departamento:
    base["Departamento_Analisis"] = (
        texto_columna(base[col_departamento])
        .str.upper()
    )
else:
    base["Departamento_Analisis"] = pd.NA

display(
    base[
        ["Empresa", "Cluster",
         "CIIU_Analisis",
         "Ciudad_Analisis",
         "Departamento_Analisis"]
    ].head(10)
)

# ================================================================
# 17. ANÁLISIS DE CIIU, CIUDADES Y DEPARTAMENTOS
# ================================================================

def analizar_categorias(columna, nombre, top_n=5):

    titulo(f"Análisis de {nombre} por clúster")

    if not base[columna].notna().any():
        print(f"No se encontraron datos de {nombre}.")
        return pd.DataFrame()

    tabla = (
        base.dropna(subset=[columna])
        .groupby(["Cluster", columna])
        .size()
        .reset_index(name="Empresas")
    )

    tabla["Porcentaje"] = (
        tabla["Empresas"]
        / tabla.groupby("Cluster")[
            "Empresas"
        ].transform("sum")*100
    )

    tabla = tabla.sort_values(
        ["Cluster", "Empresas"],
        ascending=[True, False]
    )

    display(
        tabla.groupby("Cluster")
        .head(top_n)
        .round(2)
    )

    for cluster in sorted(base["Cluster"].unique()):

        sub = (
            tabla[tabla["Cluster"] == cluster]
            .head(top_n)
            .sort_values("Empresas")
        )

        if sub.empty:
            continue

        plt.figure(figsize=(10, 5))
        barras = plt.barh(
            sub[columna].astype(str),
            sub["Empresas"],
            color=sns.color_palette(
                "Set2", len(sub)
            )
        )
        plt.bar_label(barras, padding=3)
        plt.xlim(
            0, sub["Empresas"].max()*1.2
        )
        plt.title(
            f"{nombre}: clúster {cluster}"
        )
        plt.xlabel("Número de empresas")
        plt.tight_layout()
        plt.show()

        principal = sub.iloc[-1]

        interpretar(
            observacion=(
                f"En el clúster {cluster}, "
                f"la categoría más frecuente "
                f"de {nombre.lower()} es "
                f"**{principal[columna]}**, "
                f"con {int(principal['Empresas'])} "
                f"empresas y una participación "
                f"del {principal['Porcentaje']:.2f}% "
                "entre los registros con "
                "esta información."
            ),
            significado=(
                "El resultado muestra la "
                "composición del segmento "
                "y permite identificar "
                "concentraciones empresariales."
            ),
            excelencia=(
                "El análisis contextualiza "
                "las comparaciones y ayuda "
                "a seleccionar organizaciones "
                "para benchmarking. "
                "La pertenencia a un sector "
                "o territorio no demuestra "
                "por sí sola mayor eficiencia."
            ),
            decision=(
                "Investigar prácticas, "
                "características sectoriales "
                "y condiciones del entorno "
                "antes de atribuir diferencias "
                "financieras a la operación."
            )
        )

    return tabla

tabla_ciiu = analizar_categorias(
    "CIIU_Analisis", "CIIU", 5
)

tabla_ciudades = analizar_categorias(
    "Ciudad_Analisis", "Ciudades", 5
)

tabla_departamentos = analizar_categorias(
    "Departamento_Analisis",
    "Departamentos", 8
)

# ================================================================
# 18. CLASIFICACIÓN GERENCIAL
# ================================================================

titulo("18. Clasificación gerencial de empresas")

q25 = indice.quantile(0.25)
q50 = indice.quantile(0.50)
q75 = indice.quantile(0.75)

def clasificar(valor):
    if valor >= q75:
        return "1. Referente financiero"
    if valor >= q50:
        return "2. Desempeño consolidado"
    if valor >= q25:
        return "3. Potencial de mejora"
    return "4. Atención prioritaria"

base["Clasificacion"] = (
    base["Indice_Financiero"]
    .apply(clasificar)
)

tabla_clasificacion = pd.crosstab(
    base["Cluster"],
    base["Clasificacion"]
)

display(tabla_clasificacion)

ax = tabla_clasificacion.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6),
    colormap="Set2"
)

ax.set_title(
    "Clasificación financiera por clúster"
)
ax.set_xlabel("Clúster")
ax.set_ylabel("Empresas")
ax.tick_params(axis="x", rotation=0)
ax.legend(
    title="Clasificación",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()

clasificacion_frecuente = (
    base["Clasificacion"]
    .value_counts()
    .idxmax()
)

interpretar(
    observacion=(
        "Las empresas se clasificaron "
        "en cuatro niveles de posición "
        "financiera relativa. "
        f"La categoría más frecuente es "
        f"**{clasificacion_frecuente}**."
    ),
    significado=(
        "La clasificación permite "
        "priorizar empresas para "
        "estudios comparativos "
        "y diagnósticos financieros."
    ),
    excelencia=(
        "Los referentes financieros "
        "pueden investigarse para "
        "identificar posibles buenas "
        "prácticas. Las empresas "
        "con menor posición relativa "
        "pueden requerir un análisis "
        "de desperdicios, restricciones "
        "y costos operacionales."
    ),
    decision=(
        "Diseñar programas diferenciados "
        "de diagnóstico y mejora, "
        "con indicadores operacionales "
        "que permitan evaluar resultados."
    ),
    precaucion=(
        "La clasificación es una regla "
        "basada en percentiles, "
        "no un modelo supervisado "
        "ni una evaluación de solvencia."
    )
)

# ================================================================
# 19. BENCHMARKING EMPRESARIAL
# ================================================================

titulo("19. Benchmarking empresarial")

def buscar_empresa(nombre, top_n=10):

    coincidencias = base[
        base["Empresa"].str.contains(
            nombre,
            case=False,
            regex=False,
            na=False
        )
    ]

    if coincidencias.empty:
        print("Empresa no encontrada.")
        return

    if len(coincidencias) > 1:
        print("Se encontraron varias empresas:")
        display(
            coincidencias[
                ["Empresa", "Cluster",
                 "Clasificacion"]
            ].head(20)
        )
        return

    empresa = coincidencias.iloc[0]

    print("\nEMPRESA SELECCIONADA")
    display(
        empresa[
            ["Empresa", "Cluster",
             "Clasificacion",
             "Indice_Financiero",
             "ROA", "ROE", "EBIT", "MN"]
        ].to_frame("Resultado")
    )

    # Empresas similares en los cuatro
    # indicadores financieros estandarizados
    referencia = datos_z.loc[empresa.name]

    distancias = np.sqrt(
        ((datos_z-referencia)**2).sum(axis=1)
    )

    pares = base[
        base.index != empresa.name
    ].copy()

    pares["Distancia"] = distancias.reindex(
        pares.index
    )

    print("\nEMPRESAS FINANCIERAMENTE SIMILARES")

    display(
        pares.sort_values("Distancia")[
            ["Empresa", "CIIU_Analisis",
             "Cluster", "Clasificacion",
             "Indice_Financiero", "Distancia"]
        ].head(top_n).round(3)
    )

    ciiu = empresa["CIIU_Analisis"]

    if pd.notna(ciiu):

        referentes = base[
            (base["CIIU_Analisis"] == ciiu)
            & (base.index != empresa.name)
            & (
                base["Indice_Financiero"]
                > empresa["Indice_Financiero"]
            )
        ].sort_values(
            "Indice_Financiero",
            ascending=False
        )

        print("\nREFERENTES FINANCIEROS DEL MISMO CIIU")

        if referentes.empty:
            print(
                "No se encontraron referentes "
                "con índice superior."
            )
        else:
            display(
                referentes[
                    ["Empresa", "Cluster",
                     "Indice_Financiero",
                     "ROA", "ROE", "EBIT", "MN"]
                ].head(top_n).round(3)
            )

    interpretar(
        observacion=(
            f"La empresa {empresa['Empresa']} "
            f"pertenece al clúster "
            f"{empresa['Cluster']} y tiene "
            f"un índice financiero relativo "
            f"de {empresa['Indice_Financiero']:.2f}."
        ),
        significado=(
            "La comparación permite identificar "
            "organizaciones con perfiles "
            "financieros próximos y "
            "posibles referentes sectoriales."
        ),
        excelencia=(
            "El benchmarking debe avanzar "
            "desde la comparación de resultados "
            "hacia la investigación de procesos: "
            "cómo gestionan recursos, "
            "capacidad, calidad, riesgos "
            "y mejora continua."
        ),
        decision=(
            "Seleccionar dos o tres referentes "
            "comparables y contrastar "
            "indicadores como productividad, "
            "tiempo de ciclo, desperdicios "
            "y costos de no calidad."
        )
    )

print("Empresas disponibles para consultar:")
display(
    base[
        ["Empresa", "Cluster",
         "Clasificacion"]
    ].head(10)
)

print(
    '\nEjemplo de consulta posterior:\n'
    'buscar_empresa("NOMBRE DE LA EMPRESA")'
)

# ================================================================
# 20. INFORME EJECUTIVO AUTOMÁTICO
# ================================================================

titulo("20. Informe ejecutivo de excelencia operacional")

conclusiones = []

def registrar_conclusion(texto):
    conclusiones.append(texto)
    display(Markdown(texto))

registrar_conclusion(
    f"**1. Segmentación financiera.** "
    f"Se analizaron {len(base):,} empresas "
    f"y se identificaron {mejor_k} clústeres. "
    "La segmentación revela perfiles "
    "financieros diferenciados que pueden "
    "servir como punto de partida "
    "para el benchmarking."
)

registrar_conclusion(
    f"**2. Reducción de dimensionalidad.** "
    f"Las dos primeras componentes PCA "
    f"conservan {varianza_2:.2f}% "
    "de la variabilidad de los indicadores "
    "estandarizados. Esto facilita "
    "visualizar patrones empresariales "
    "y comunicar resultados analíticos."
)

registrar_conclusion(
    f"**3. Posición financiera relativa.** "
    f"El clúster {lider} presenta "
    "el mayor índice financiero "
    "promedio de la muestra. "
    "Sus empresas pueden ser candidatas "
    "para estudiar prácticas gerenciales, "
    "sin asumir que sus resultados "
    "demuestran excelencia operacional."
)

for indicador in INDICADORES:

    mejor = medias[indicador].idxmax()
    menor = medias[indicador].idxmin()
    brecha = (
        medias[indicador].max()
        - medias[indicador].min()
    )

    registrar_conclusion(
        f"**Brecha de {indicador}.** "
        f"El clúster {mejor} presenta "
        f"el mayor promedio y el clúster "
        f"{menor} el menor. "
        f"La diferencia es {brecha:.4f}. "
        "Se recomienda investigar "
        "si existen factores operacionales "
        "asociados a esta diferencia."
    )

registrar_conclusion(
    "**4. Generación de valor.** "
    "Los resultados pueden orientar "
    "decisiones sobre priorización "
    "de proyectos, diagnóstico de costos, "
    "utilización de activos, productividad "
    "y mejora continua. "
    "Para demostrar generación de valor "
    "se necesitan datos adicionales "
    "sobre inversiones, costos, "
    "riesgos y resultados operacionales."
)

registrar_conclusion(
    "**5. Recomendación estratégica.** "
    "Combinar el análisis financiero "
    "con herramientas de excelencia "
    "operacional como Lean, DMAIC, "
    "VSM, FMEA, análisis de capacidad "
    "y evaluación multicriterio AHP."
)

# ================================================================
# 21. MATRIZ DE DECISIONES GERENCIALES
# ================================================================

titulo("21. Matriz de decisiones desde excelencia operacional")

recomendaciones = pd.DataFrame([
    {
        "Dimensión": "Productividad",
        "Pregunta": "¿Se aprovechan adecuadamente los recursos?",
        "Herramienta": "VSM y análisis de capacidad",
        "Indicador": "Productividad por recurso",
        "Valor": "Mayor rendimiento de recursos"
    },
    {
        "Dimensión": "Costos",
        "Pregunta": "¿Dónde se generan desperdicios?",
        "Herramienta": "Lean y Pareto",
        "Indicador": "Costo por unidad o servicio",
        "Valor": "Reducción de costos"
    },
    {
        "Dimensión": "Calidad",
        "Pregunta": "¿Cuánto cuesta la no calidad?",
        "Herramienta": "DMAIC y Six Sigma",
        "Indicador": "Defectos y reprocesos",
        "Valor": "Menor variabilidad"
    },
    {
        "Dimensión": "Capacidad",
        "Pregunta": "¿Dónde aparecen restricciones?",
        "Herramienta": "Teoría de restricciones",
        "Indicador": "Utilización y tiempos de espera",
        "Valor": "Mayor flujo operacional"
    },
    {
        "Dimensión": "Riesgos",
        "Pregunta": "¿Qué eventos amenazan el desempeño?",
        "Herramienta": "FMEA",
        "Indicador": "Riesgos críticos",
        "Valor": "Resiliencia"
    },
    {
        "Dimensión": "Sostenibilidad",
        "Pregunta": "¿Cómo reducir el uso de recursos?",
        "Herramienta": "Economía circular",
        "Indicador": "Intensidad de recursos",
        "Valor": "Valor sostenible"
    },
    {
        "Dimensión": "Estrategia",
        "Pregunta": "¿Qué proyectos generan mayor valor?",
        "Herramienta": "AHP y evaluación financiera",
        "Indicador": "ROI y ROIC",
        "Valor": "Mejor asignación de capital"
    }
])

display(recomendaciones)

# ================================================================
# 22. EXPORTAR RESULTADOS
# ================================================================

titulo("22. Exportación del laboratorio")

nombre_salida = (
    "PCA_KMeans_Excelencia_Operacional_Interpretado.xlsx"
)

informe = pd.DataFrame({
    "Conclusiones": conclusiones
})

with pd.ExcelWriter(
    nombre_salida,
    engine="openpyxl"
) as writer:

    base.to_excel(
        writer,
        sheet_name="Empresas_Clasificadas",
        index=False
    )

    estadisticas.to_excel(
        writer,
        sheet_name="Estadisticas"
    )

    corr.to_excel(
        writer,
        sheet_name="Correlaciones"
    )

    datos_z.to_excel(
        writer,
        sheet_name="Estandarizados"
    )

    tabla_varianza.to_excel(
        writer,
        sheet_name="Varianza_PCA",
        index=False
    )

    cargas.to_excel(
        writer,
        sheet_name="Cargas_PCA"
    )

    resultado.to_excel(
        writer,
        sheet_name="PCA_Clusters"
    )

    tabla_k.to_excel(
        writer,
        sheet_name="Seleccion_K",
        index=False
    )

    medias.to_excel(
        writer,
        sheet_name="Promedios_Clusters"
    )

    medianas.to_excel(
        writer,
        sheet_name="Medianas_Clusters"
    )

    medias_z.to_excel(
        writer,
        sheet_name="Perfiles_Estandarizados"
    )

    tabla_clasificacion.to_excel(
        writer,
        sheet_name="Clasificacion"
    )

    if not tabla_ciiu.empty:
        tabla_ciiu.to_excel(
            writer,
            sheet_name="CIIU",
            index=False
        )

    if not tabla_ciudades.empty:
        tabla_ciudades.to_excel(
            writer,
            sheet_name="Ciudades",
            index=False
        )

    if not tabla_departamentos.empty:
        tabla_departamentos.to_excel(
            writer,
            sheet_name="Departamentos",
            index=False
        )

    recomendaciones.to_excel(
        writer,
        sheet_name="Recomendaciones",
        index=False
    )

    informe.to_excel(
        writer,
        sheet_name="Informe_Ejecutivo",
        index=False
    )

print("\n" + "="*65)
print("LABORATORIO FINALIZADO CORRECTAMENTE")
print("="*65)
print("Archivo:", nombre_archivo)
print("Empresas analizadas:", len(base))
print("Clústeres:", mejor_k)
print(f"Varianza PCA: {varianza_2:.2f}%")
print("Resultados:", nombre_salida)

files.download(nombre_salida)
